
# Exercises XP Ninja — **Monitoring NLP Models in Production** (Solution)


This notebook completes the XP Ninja challenge end-to-end:

1. **Dataset & Setup** (drug reviews if available; else synthetic)  
2. **Baseline Sentiment Model** (TF‑IDF + Logistic Regression)  
3. **Quality Issues Simulation** (label noise, text corruption, missing/punctuation, casing)  
4. **Distribution Drift Simulation** (topic/lexicon shift, OOV rate)  
5. **Monitoring & Reports with Evidently** (data drift & classification performance)  
6. **Quality Decay Analysis & Remediation Ideas**  
7. **Short Recap** ready to share publicly

> Notes
> - Matplotlib for charts (no seaborn).
> - The code tries to load a drug reviews CSV if present; otherwise synthesizes a realistic text dataset.
> - Evidently is installed within the notebook when you run it in Colab/Jupyter.



## 1) Setup & Data Loading

- Attempt to load a **drug reviews** dataset from common paths.  
- If not found, create a **synthetic sentiment dataset** with domain-like vocabulary.  
- Split into **train/validation/test** with stratification.


In [ ]:

import os, re, random, string, math, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix
)

random.seed(42); np.random.seed(42)

# Try to find a "drug reviews" dataset if the user provided one.
CANDIDATES = [
    '/mnt/data/drug_reviews.csv',
    '/mnt/data/drug_review.csv',
    '/mnt/data/drugsComTrain_raw.tsv',  # kaggle-style
    '/mnt/data/drugsComTest_raw.tsv',
    'drug_reviews.csv', 'drug_review.csv'
]

def load_drug_reviews():
    for p in CANDIDATES:
        if os.path.exists(p):
            if p.endswith('.tsv'):
                df = pd.read_csv(p, sep='\t')
            else:
                df = pd.read_csv(p)
            # Heuristic mapping
            lower_cols = {c.lower(): c for c in df.columns}
            text_col = lower_cols.get('review') or lower_cols.get('text') or list(df.columns)[0]
            # Rating to sentiment label if available (>=7 positive)
            if 'rating' in lower_cols:
                rating_col = lower_cols['rating']
                y = (df[rating_col] >= 7).astype(int)
            else:
                # Otherwise try "sentiment"/"label"
                sent_col = lower_cols.get('sentiment') or lower_cols.get('label')
                if sent_col:
                    y = (df[sent_col].astype(str).str.lower().str.contains('pos|1')).astype(int)
                else:
                    # If nothing usable, bail
                    return None
            return pd.DataFrame({'text': df[text_col].astype(str), 'label': y})
    return None

def synthesize_reviews(n=8000):
    pos_symptoms = ["pain relief","sleep improved","no side effects","energy up","clear skin","appetite normal"]
    neg_symptoms = ["nausea","headache","rash","dizziness","fatigue","insomnia","stomach pain"]
    pos_words = ["effective","amazing","helped","great","recommended","works","relief","improved","happy","better"]
    neg_words = ["terrible","awful","worse","didn't work","useless","bad","side effects","disappointed","anxious","hurt"]
    drug_names = ["Ibuproxin","Acetamed","Sertalin","Metforal","Lorazil","Amoxicin","Omepraz","Isotret","Zyrtin","Morphal"]
    conditions = ["migraine","anxiety","acne","diabetes","reflux","infection","allergy","pain","insomnia","depression"]
    templates_pos = [
        "After taking {drug} for {cond}, I feel {posw}. {symp}.",
        "{drug} really {posw} my {cond}. {symp}.",
        "My doctor suggested {drug}; results were {posw}. {symp}.",
    ]
    templates_neg = [
        "I used {drug} for {cond} and it was {negw}. Got {symp}.",
        "{drug} made my {cond} {negw}. Also had {symp}.",
        "Tried {drug}; {negw} experience. {symp}.",
    ]
    rng = np.random.default_rng(42)
    rows = []
    for i in range(n):
        if rng.random() < 0.5:
            txt = random.choice(templates_pos).format(
                drug=random.choice(drug_names),
                cond=random.choice(conditions),
                posw=random.choice(pos_words),
                symp=random.choice(pos_symptoms)
            )
            y = 1
        else:
            txt = random.choice(templates_neg).format(
                drug=random.choice(drug_names),
                cond=random.choice(conditions),
                negw=random.choice(neg_words),
                symp=random.choice(neg_symptoms)
            )
            y = 0
        rows.append((txt, y))
    return pd.DataFrame(rows, columns=['text','label'])

df = load_drug_reviews()
if df is None:
    df = synthesize_reviews(8000)

# Basic clean
df['text'] = df['text'].fillna('')

# Split
train_df, temp_df = train_test_split(df, test_size=0.4, random_state=42, stratify=df['label'])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['label'])

print('Train/Val/Test sizes:', len(train_df), len(val_df), len(test_df))
display(df.head())



## 2) Baseline Sentiment Classifier (TF‑IDF + Logistic Regression)

- Build a scikit‑learn `Pipeline` with `TfidfVectorizer` + `LogisticRegression`.  
- Evaluate on **validation** and **test** sets.  
- Keep this as the **production candidate**.


In [ ]:

model = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=30000,
        ngram_range=(1,2),
        min_df=2
    )),
    ('clf', LogisticRegression(max_iter=1000, n_jobs=None, random_state=42))
])

model.fit(train_df['text'], train_df['label'])

def evaluate(split_name, X, y):
    proba = model.predict_proba(X)[:,1]
    preds = (proba >= 0.5).astype(int)
    acc = accuracy_score(y, preds)
    prec = precision_score(y, preds)
    rec = recall_score(y, preds)
    f1 = f1_score(y, preds)
    roc = roc_auc_score(y, proba)
    print(f'[{split_name}] Acc={acc:.4f}  Prec={prec:.4f}  Rec={rec:.4f}  F1={f1:.4f}  ROC-AUC={roc:.4f}')
    return {'acc':acc,'prec':prec,'rec':rec,'f1':f1,'roc':roc}

m_val = evaluate('Validation', val_df['text'], val_df['label'])
m_test = evaluate('Test', test_df['text'], test_df['label'])

# Confusion matrix (matplotlib)
cm = confusion_matrix(test_df['label'], (model.predict_proba(test_df['text'])[:,1] >= 0.5).astype(int))
plt.figure(figsize=(4,4))
plt.imshow(cm)
plt.title('Confusion Matrix — Test')
plt.xlabel('Predicted')
plt.ylabel('Actual')
for (i,j), v in np.ndenumerate(cm):
    plt.text(j, i, str(v), ha='center', va='center')
plt.tight_layout()
plt.show()



## 3) Simulating **Data Quality Issues**

We create perturbed copies of the **test** split:
- **Label noise** (flip a fraction of labels)
- **Punctuation stripping & lowercasing**
- **Unicode corruption** (random character drops/inserts)
- **Missing text** (empty strings for a fraction)


In [ ]:

def add_label_noise(y, frac=0.15, seed=42):
    rng = np.random.default_rng(seed)
    y_noisy = y.copy()
    idx = rng.choice(len(y), size=int(frac*len(y)), replace=False)
    y_noisy.iloc[idx] = 1 - y_noisy.iloc[idx]
    return y_noisy

def strip_punct_and_lower(texts):
    table = str.maketrans('', '', string.punctuation)
    return texts.apply(lambda s: s.translate(table).lower())

def unicode_corrupt(texts, drop_prob=0.05, seed=42):
    rng = np.random.default_rng(seed)
    def corrupt(s):
        out = []
        for ch in s:
            if rng.random() < drop_prob:
                # drop or random insert
                if rng.random() < 0.5:
                    continue
                else:
                    out.append(random.choice(['�','¤','§','~']))
            out.append(ch)
        return ''.join(out)
    return texts.apply(corrupt)

def introduce_missing(texts, frac=0.1, seed=42):
    rng = np.random.default_rng(seed)
    texts2 = texts.copy()
    idx = rng.choice(len(texts2), size=int(frac*len(texts2)), replace=False)
    texts2.iloc[idx] = ''
    return texts2

variants = {
    'baseline': (test_df['text'], test_df['label']),
    'label_noise_15': (test_df['text'], add_label_noise(test_df['label'], 0.15)),
    'punct_lower': (strip_punct_and_lower(test_df['text']), test_df['label']),
    'unicode_corrupt': (unicode_corrupt(test_df['text']), test_df['label']),
    'missing_10pct': (introduce_missing(test_df['text'], 0.10), test_df['label']),
}

quality_metrics = {}
for name, (Xv, yv) in variants.items():
    print('\nVariant:', name)
    quality_metrics[name] = evaluate(name, Xv, yv)

# Plot F1 drop vs baseline
baseline_f1 = quality_metrics['baseline']['f1']
labels = list(quality_metrics.keys())
f1s = [quality_metrics[k]['f1'] for k in labels]
plt.figure(figsize=(6,4))
plt.bar(range(len(labels)), f1s)
plt.xticks(range(len(labels)), labels, rotation=30, ha='right')
plt.ylabel('F1 Score')
plt.title('F1 across Quality Variants')
plt.tight_layout()
plt.show()

print('\nΔF1 vs baseline:')
for k in labels:
    print(f'  {k:15s}: {f1s[labels.index(k)] - baseline_f1:+.4f}')



## 4) Simulating **Distribution Drift**

We create a **drifted dataset** by shifting lexicon/phrases:
- Positive words replaced with neutral/unknown tokens.
- New negative synonyms injected.
We then evaluate performance to observe **quality decay**.


In [ ]:

# Create drift by replacing positive phrases and injecting new negatives
pos_tokens = ["effective","amazing","helped","great","recommended","works","relief","improved","happy","better"]
new_neg_tokens = ["underwhelming","underperform","meh","mediocre","questionable","buggy","sideeffecty"]

def drift_texts(texts):
    def drift_line(s):
        s2 = s
        for w in pos_tokens:
            s2 = re.sub(rf'\b{re.escape(w)}\b', 'UNK', s2, flags=re.IGNORECASE)
        if random.random() < 0.3:
            s2 += " " + random.choice(new_neg_tokens)
        return s2
    return texts.apply(drift_line)

drift_text = drift_texts(test_df['text'])
drift_metrics = evaluate('drifted_texts', drift_text, test_df['label'])



## 5) Monitoring with **Evidently**

We generate:
- **Data drift** report on text features (TF‑IDF space proxy via vector norms & length).
- **Classification performance** dashboard comparing **reference** (validation) vs **production** (current test/drift).
> In real setups, you would log raw text + predictions + metadata, then feed to Evidently with well-defined schemas.


In [ ]:

# Install Evidently if missing (works in Colab/Jupyter)
import sys, subprocess, importlib

def ensure_pkg(pkg):
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure_pkg("evidently")

from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, ClassificationPreset

# Build a simple monitoring DataFrame for reference vs production
def build_monitor_frame(df_split, split_name):
    proba = model.predict_proba(df_split['text'])[:,1]
    pred = (proba >= 0.5).astype(int)
    return pd.DataFrame({
        'text': df_split['text'].values,
        'label': df_split['label'].values,
        'pred': pred,
        'proba': proba,
        'split': split_name,
        'length': df_split['text'].str.len().values
    })

ref_df = build_monitor_frame(val_df, 'reference')
prod_df = build_monitor_frame(test_df, 'production')
drift_df = pd.DataFrame({'text': drift_text, 'label': test_df['label']})
drift_prod_df = build_monitor_frame(drift_df, 'production_drift')

# Data Drift: reference vs production (using simple proxies like length)
data_drift_report = Report(metrics=[DataDriftPreset()])
data_drift_report.run(reference_data=ref_df[['length']], current_data=prod_df[['length']])
data_drift_path = '/mnt/data/evidently_data_drift.html'
data_drift_report.save_html(data_drift_path)

# Classification performance: reference vs production
classif_report = Report(metrics=[ClassificationPreset()])
classif_report.run(reference_data=ref_df[['label','pred','proba']],
                   current_data=prod_df[['label','pred','proba']])
classif_path = '/mnt/data/evidently_classif_prod.html'
classif_report.save_html(classif_path)

# Classification performance with drifted texts
classif_drift_report = Report(metrics=[ClassificationPreset()])
classif_drift_report.run(reference_data=ref_df[['label','pred','proba']],
                         current_data=drift_prod_df[['label','pred','proba']])
classif_drift_path = '/mnt/data/evidently_classif_prod_drift.html'
classif_drift_report.save_html(classif_drift_path)

print('Saved reports:')
print(' -', data_drift_path)
print(' -', classif_path)
print(' -', classif_drift_path)



## 6) Quality Decay Analysis & Remediation

**Signals observed (typical):**
- Drops in **F1/ROC-AUC** under label noise and drift.
- Changes in **text length** or **token distributions** (proxy) indicating input drift.
- Higher **false negatives** when positive lexicon is replaced by OOV/UNK tokens.

**Mitigations:**
1. **Data QA gates**: reject/flag empty or corrupted text; minimum length checks.  
2. **Lexicon augmentation**: expand TF‑IDF vocabulary / use subword models (e.g., fastText) to reduce OOV.  
3. **Regular re-training** with fresh labels from recent production data.  
4. **Threshold tuning** per segment (e.g., condition/drug).  
5. **Human-in-the-loop** review for high-uncertainty predictions.  
6. **Logging & dashboards**: centralize Evidently HTML reports; alert on drift/quality decay thresholds.



## 7) Short Recap (Copy‑Paste for Social/DI)

> Built a TF‑IDF + Logistic Regression sentiment model on drug‑review‑like data, then simulated **data quality issues** (label noise, punctuation/Unicode corruption, missing text) and **distribution drift** (lexicon shift). Evaluated quality decay (F1/ROC‑AUC), and generated **Evidently** monitoring reports for **data drift** and **classification performance** comparing reference vs production vs drifted streams. Proposed mitigation steps (data gates, re‑training, lexicon augmentation, threshold tuning, HITL). Reports exported as HTML for easy sharing.
